# Phase 0: Kaggle environment probe

Disposable, throwaway notebook. Its ONLY job is to answer, before anything else
is built: **does real ALFWorld install and run here, and does the chosen local
model load and generate?** Nothing here produces a real result -- it produces a
pass/fail report per the approved plan
(`C:\Users\Ibo\.claude\plans\logical-meandering-feather.md`).

Settings this notebook needs: **GPU accelerator ON** (T4 x2 or P100), **Internet ON**
(see the plan for the offline-dataset fallback if this account/notebook type
can't enable internet).

## 0. Basic environment facts (Python version, GPU, disk)

In [ ]:
import sys, subprocess, shutil
print('Python:', sys.version)
print()
subprocess.run(['nvidia-smi'])
print()
print('Disk free on /kaggle/working:', shutil.disk_usage('/kaggle/working'))
print('gcc present:', shutil.which('gcc'))
print('cmake present:', shutil.which('cmake'))

## 1. ALFWorld install -- THE make-or-break step

The paid run's own history: native Windows failed outright (no C toolchain);
even on Linux, `textworld` 1.7.0 needed Python 3.11 specifically (a Python
3.13 incompatibility via PEP 667). Kaggle's default Python is unverified for
this -- if this cell fails, STOP and report back before building anything else
(see the plan's Phase 0 risk note: don't fall back to a mock env, since that
would defeat the point of this replication).

In [ ]:
!pip install alfworld tqdm termcolor --quiet
import alfworld
print('alfworld imported OK, version:', getattr(alfworld, '__version__', 'unknown'))

In [ ]:
# ~2.3GB download -- confirms Internet ON is actually working, and gets the
# real game/PDDL data + grammar files into ~/.cache/alfworld.
!alfworld-download

In [ ]:
# One real env.reset()/env.step() -- if this cell runs without error and
# prints a real-looking observation + admissible commands, ALFWorld itself
# works on Kaggle. Uses alfworld's official base_config.yaml directly (fetch
# it from the official repo if not already present).
import os, urllib.request
if not os.path.exists('base_config.yaml'):
    urllib.request.urlretrieve(
        'https://raw.githubusercontent.com/alfworld/alfworld/master/configs/base_config.yaml',
        'base_config.yaml',
    )

import yaml
from alfworld.agents.environment import get_environment

with open('base_config.yaml') as f:
    cfg = yaml.safe_load(f)

env_type = cfg['env']['type']
env = get_environment(env_type)(cfg, train_eval='train')
env = env.init_env(batch_size=1)
obs, info = env.reset()
print('PASS: real env.reset() succeeded')
print('obs:', obs[0][:300])
print('admissible_commands (first 5):', info['admissible_commands'][0][:5])
print('gamefile:', info['extra.gamefile'][0])

obs2, scores, dones, infos2 = env.step([info['admissible_commands'][0][0]])
print('PASS: real env.step() succeeded')
print('obs2:', obs2[0][:300])
env.close()

## 2. Local model: try vLLM first, fall back to `transformers`

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
REVISION = 'a09a35458c702b33eeacc393d103063234e8bc28'  # confirmed exact commit at plan time -- re-verify before a real run

vllm_works = False
try:
    get_ipython().system('pip install vllm --quiet')
    from vllm import LLM, SamplingParams
    llm = LLM(model=MODEL_ID, revision=REVISION, dtype='float16', gpu_memory_utilization=0.85)
    out = llm.generate(['Say hello in one short sentence.'], SamplingParams(temperature=0, max_tokens=20))
    print('vLLM PASS:', out[0].outputs[0].text)
    vllm_works = True
except Exception as e:
    print('vLLM did not work cleanly, falling back to transformers. Error was:')
    print(repr(e))

print()
print('vllm_works =', vllm_works, '-- record this in kaggle_config.yaml (llm.backend)')

In [ ]:
# Only run this cell if vllm_works is False above.
if not vllm_works:
    get_ipython().system('pip install transformers accelerate bitsandbytes --quiet')
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tok = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type='nf4')
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, torch_dtype=torch.float16, device_map='cuda', quantization_config=quant)
    model.eval()

    messages = [{'role': 'user', 'content': 'Say hello in one short sentence.'}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tok.eos_token_id)
    text = tok.decode(out_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print('transformers PASS:', text)

    print()
    print('Peak VRAM used (GB):', torch.cuda.max_memory_allocated() / 1e9)

## 3. Report

Fill this in by hand after running the cells above, then report back before
Phase 1 is scoped in detail:

- [ ] ALFWorld install: PASS / FAIL (paste error if FAIL)
- [ ] `alfworld-download`: PASS / FAIL
- [ ] Real `env.reset()`/`env.step()`: PASS / FAIL
- [ ] vLLM: PASS / FAIL -- if FAIL, transformers fallback: PASS / FAIL
- [ ] Peak VRAM at 4-bit: ____ GB (compare against the ~16GB T4/P100 budget)
- [ ] Python version actually running: ____
- [ ] gcc/cmake present by default: ____